In [ ]:
import pandas as pd
import json
import os
import psycopg2
from dotenv import load_dotenv

##### Connection Neon

In [ ]:
load_dotenv()

DATABASE_URL = os.getenv("NEON_DATABASE_URL")

if not DATABASE_URL:
    raise ValueError("La variable NEON_DATABASE_URL n'est pas définie dans .env")

try:
    # On ajoute sslmode=require pour Neon
    conn = psycopg2.connect(DATABASE_URL)
    cur = conn.cursor()
    
    # Test pour confirmer la version de la BDD
    cur.execute("SELECT version();")
    db_version = cur.fetchone()
    print(f"✅ Connexion à Néon réussie !")
    print(f"🖥️ Version : {db_version[0][:30]}...")

except Exception as e:
    print(f"❌ Erreur de connexion : {e}")

#### Visu de Neon

In [ ]:
query = "SELECT * FROM bilans_entreprises;" 

try:
    # On utilise la connexion 'conn' que tu as déjà établie
    df = pd.read_sql_query(query, conn)
    print(f"📊 Données chargées avec succès !")
    print(f"Lignes : {df.shape[0]} | Colonnes : {df.shape[1]}")
    
    # On regarde les premières colonnes pour identifier celle qui contient le JSON
    print("\nAperçu des colonnes :")
    print(df.columns.tolist())
    
    # On affiche un aperçu
    display(df.head(3))
    
except Exception as e:
    print(f"❌ Erreur lors de la lecture : {e}")

---

In [ ]:
# 1. Sécurité : On s'assure que 'donnees_liasses' est bien un dictionnaire Python
def ensure_dict(val):
    if isinstance(val, dict): return val
    if isinstance(val, str):
        try:
            # On tente de parser le JSON (gère les guillemets doubles)
            return json.loads(val.replace("'", '"')) 
        except:
            try:
                # Si ça échoue, on tente l'évaluation littérale (gère les guillemets simples)
                import ast
                return ast.literal_eval(val)
            except:
                return {}
    return {}

print("🔄 Préparation de la colonne JSON...")
df['donnees_liasses'] = df['donnees_liasses'].apply(ensure_dict)

# 2. La fonction "Universal Extractor" (Celle qui t'a donné 82%)
def universal_extractor(row):
    d = row.get('donnees_liasses', {})
    if not isinstance(d, dict) or not d: 
        return 0.0, 0.0, 0.0

    # ÉTAPE A : Si structure imbriquée 'bilanSaisi' (le fameux labyrinthe), on l'aplatit
    if 'bilanSaisi' in d:
        try:
            pages = d.get('bilanSaisi', {}).get('bilan', {}).get('detail', {}).get('pages', [])
            flat_d = {}
            for page in pages:
                for liasse in page.get('liasses', []):
                    code = liasse.get('code')
                    # On prend la valeur m1 (net/brut) ou m3 (amortissements)
                    val = liasse.get('m1') or liasse.get('m3')
                    if code and val is not None: 
                        flat_d[code] = val
            # Si on a réussi à extraire des codes, on remplace d par cette version plate
            if flat_d:
                d = flat_d 
        except: pass

    # ÉTAPE B : Extraction avec les codes prioritaires et alternatifs
    # TOTAL ACTIF (Normal: AT, Simplifié: 210, Brut: BJ, Immo/Passif: DL, 232)
    actif = d.get('AT') or d.get('210') or d.get('BJ') or d.get('DL') or d.get('BP') or d.get('232') or 0
    
    # CHIFFRE D'AFFAIRES (Normal: EE, Simplifié: 218, Brut/Ventes: 0G, 010, FL)
    ca = d.get('EE') or d.get('218') or d.get('0G') or d.get('010') or d.get('FL') or 0
    
    # RÉSULTAT NET (DI, 310, HN, 120)
    res = d.get('DI') or d.get('310') or d.get('HN') or d.get('120') or 0
    
    return float(actif), float(ca), float(res)

# 3. Application du traitement
print("⏳ Extraction en profondeur sur 3289 lignes...")
extracted_values = df.apply(universal_extractor, axis=1)

# On crée les colonnes propres
df[['total_actif', 'ca', 'resultat_net']] = pd.DataFrame(extracted_values.tolist(), index=df.index)

# 4. Diagnostic Final
taux_actif = (df['total_actif'] > 0).mean() * 100
taux_ca = (df['ca'] > 0).mean() * 100

print(f"\n✅ Diagnostic de succès :")
print(f"📈 Taux de récupération Actif : {taux_actif:.2f}%")
print(f"📈 Taux de récupération CA    : {taux_ca:.2f}%")
print(f"🚀 Entreprises exploitables   : {len(df[df['total_actif'] > 0])}")

# Aperçu du résultat
display(df[df['total_actif'] > 0][['siren', 'date_cloture', 'total_actif', 'ca', 'resultat_net']].head(10))

In [ ]:
# Cellule d'inspection complète sans filtre
import numpy as np

# On ajoute des colonnes de diagnostic temporaires
df['is_empty'] = df['donnees_liasses'].apply(lambda d: len(d) == 0 if isinstance(d, dict) else True)
df['has_bilanSaisi'] = df['donnees_liasses'].apply(lambda d: 'bilanSaisi' in d if isinstance(d, dict) else False)

# Sélection des colonnes pour l'inspection
cols_to_show = [
    'siren', 
    'date_cloture', 
    'total_actif', 
    'ca', 
    'resultat_net', 
    'confidentialite', 
    'is_empty', 
    'has_bilanSaisi'
]

# On affiche un échantillon mixte (mélange de succès et d'échecs)
# pour voir à quoi ressemblent les lignes à 0
inspection_df = df[cols_to_show].sort_values(by='total_actif', ascending=True)

print(f"🔍 INSPECTION GLOBALE ({len(df)} lignes)")
display(inspection_df.head(20)) # On regarde les premiers (souvent les 0.0)

In [ ]:
# 1. Correction définitive des signes pour l'Actif et le CA
df['total_actif'] = df['total_actif'].abs()
df['ca'] = df['ca'].abs()

# 2. Création de Ratios "Spécial Faillite"
# Un résultat négatif sur un petit CA est plus grave que sur un gros CA
df['rentabilite'] = df['resultat_net'] / df['ca'].replace(0, 1) # Évite division par 0

# Le poids de la structure par rapport à son activité
df['intensite_actif'] = df['total_actif'] / df['ca'].replace(0, 1)

# 3. Marquage manuel du cas test (en attendant ton fichier complet)
# Juste pour voir si le profil change
check_siren = "422362426"
display(df[df['siren'] == check_siren][['siren', 'date_cloture', 'total_actif', 'ca', 'resultat_net', 'rentabilite']])

In [ ]:
# On compte combien de lignes ont un JSON qui n'est pas '{}' ou NULL
query_count = """
SELECT count(*) 
FROM bilans_entreprises 
WHERE donnees_liasses IS NOT NULL 
  AND donnees_liasses::text != '{}';
"""
count_reel = pd.read_sql(query_count, conn).iloc[0,0]
print(f"📈 Nombre total de bilans potentiellement exploitables en base : {count_reel}")

#### Extraction des bilans pour augmentation de l'échantillon

In [ ]:
# # 1. Requête pour extraire TOUT le gisement exploitable
# query_extract_all = """
# SELECT siren, date_cloture, donnees_liasses, confidentialite
# FROM bilans_entreprises 
# WHERE donnees_liasses IS NOT NULL 
#   AND donnees_liasses::text != '{}'
#   AND donnees_liasses::text != 'null';
# """

# print(f"🚀 Extraction de {count_reel} lignes en cours...")
# df_full = pd.read_sql(query_extract_all, conn)

# # 2. Sécurisation du format JSON (ton ancienne fonction ensure_dict)
# df_full['donnees_liasses'] = df_full['donnees_liasses'].apply(ensure_dict)

# # 3. Extraction des variables financières avec ton Universal Extractor
# print("⏳ Transformation des données JSON en colonnes financières...")
# extracted_data = df_full.apply(universal_extractor, axis=1)
# df_full[['total_actif', 'ca', 'resultat_net']] = pd.DataFrame(extracted_data.tolist(), index=df_full.index)

# # 4. Nettoyage final et correction des signes
# # On ne garde que ceux qui ont un actif (preuve de réussite de l'extraction)
# df_ready = df_full[df_full['total_actif'] != 0].copy()
# df_ready['total_actif'] = df_ready['total_actif'].abs()
# df_ready['ca'] = df_ready['ca'].abs()

# # 5. Sauvegarde locale pour ton futur modèle
# df_ready.to_csv('dataset_bilans_complet.csv', index=False)

# print(f"✅ Terminé ! {len(df_ready)} bilans exploitables sauvegardés dans 'dataset_bilans_complet.csv'")
